In [ ]:
import gymnasium as gym
from tqdm import tqdm


# Initialise the environment
n_envs = 8
n_updates = 7500
n_rollout = 10


envs = gym.make_vec("LunarLander-v3", num_envs=n_envs, max_episode_steps=600,
                    continuous=False, gravity=-10.0, enable_wind=False,
                    wind_power=15.0, turbulence_power=1.5)

envs_wrapper = gym.wrappers.vector.RecordEpisodeStatistics(
  envs, buffer_length=n_envs*n_updates
)

obs_shape = int(envs.single_observation_space.shape[0])
action_shape = int(envs.single_action_space.n)

In [ ]:
import torch as t
import torch.nn as nn
import torch.nn.functional as F

from typing import Annotated

class A2C(nn.Module):
  def __init__(self, obs_shape: int, action_shape: int, device: str = "cuda", critic_lr: float = 3e-3, actor_lr: float = 3e-3,):
    super().__init__()
    self.device = device
    self._obs_shape = obs_shape
    self._action_shape = action_shape

    # pi(a|s)
    self.actor = nn.Sequential(
      nn.Linear(self._obs_shape, 32),
      nn.ReLU(),
      nn.Linear(32, 32),
      nn.ReLU(),
      nn.Linear(32, self._action_shape),
    )

    # V(s)
    self.critic = nn.Sequential(
      nn.Linear(self._obs_shape, 32),
      nn.ReLU(),
      nn.Linear(32, 32),
      nn.ReLU(),
      nn.Linear(32, 1),
    )

    self.actor_optim = t.optim.Adam(self.actor.parameters(), lr=actor_lr)
    self.critic_optim = t.optim.Adam(self.critic.parameters(), lr=critic_lr)

  def forward(self, obs: Annotated[t.Tensor, "n_envs obs_shape"]):
    obs.to(self.device)

    action_logits = self.actor(obs)
    state_values = self.critic(obs)

    return action_logits, state_values


  def select_action(self, obs: Annotated[t.Tensor, "n_envs obs_shape"]) -> tuple[
    Annotated[t.Tensor, "n_envs"],
    Annotated[t.Tensor, "n_envs"],
    Annotated[t.Tensor, "n_envs"],
    Annotated[t.Tensor, "n_envs"]
  ]:
    action_logits, state_values = self.forward(obs)
    action_dist = t.distributions.Categorical(logits=action_logits)
    actions = action_dist.sample()
    action_log_probs = action_dist.log_prob(actions)

    return actions, action_log_probs, state_values, action_dist.entropy()

  def update_params(self, actor_loss: t.Tensor, critic_loss: t.Tensor):    
    self.actor_optim.zero_grad()
    self.critic_optim.zero_grad()

    actor_loss.backward()
    critic_loss.backward()

    nn.utils.clip_grad_norm_(self.actor.parameters(), 0.5)
    nn.utils.clip_grad_norm_(self.critic.parameters(), 0.5)

    self.actor_optim.step()
    self.critic_optim.step()

In [ ]:
from collections import deque
import numpy as np

df = 0.99
lr = 7e-4  # this is alpha = step_size
device = "cuda"

a2c = A2C(obs_shape, action_shape, device, lr, lr)
a2c.to(device)

scheds = [t.optim.lr_scheduler.LinearLR(o, 1.0, 0.0, n_updates)
          for o in (a2c.actor_optim, a2c.critic_optim)]

obs, info = envs.reset()
obs = t.tensor(obs, device=device)


pbar = tqdm(range(n_updates))
recent_returns = deque(maxlen=100)

for update in pbar:
  # Episode:
  ep_state_values = t.zeros(n_rollout, n_envs, device=device)
  ep_action_log_probs = t.zeros(n_rollout, n_envs, device=device)
  ep_rewards = t.zeros(n_rollout, n_envs, device=device)
  ep_entropy = t.zeros(n_rollout, n_envs, device=device)
  masks = t.zeros(n_rollout, n_envs, device=device)

  for step in range(n_rollout):
    actions, action_log_probs, state_values, entropy = a2c.select_action(obs)

    states, rewards, terminated, truncated, infos = envs_wrapper.step(actions.cpu().numpy())

    if "episode" in infos:
      ep_done = infos["_episode"]
      recent_returns.extend(infos["episode"]["r"][ep_done].tolist())

    ep_state_values[step] = state_values.squeeze()
    ep_rewards[step] = t.tensor(rewards, device=device)
    ep_action_log_probs[step] = action_log_probs
    ep_entropy[step] = entropy
    
    masks[step] = t.tensor(~(terminated | truncated), dtype=t.float32, device=device)
    
    obs = t.tensor(states, dtype=t.float32, device=device)

    # a time limit isn't a terminal state, so bootstrap it back into the reward
    if truncated.any():
      with t.no_grad():
        ep_rewards[step] += df * a2c.critic(obs).squeeze(-1) * t.tensor(truncated, dtype=t.float32, device=device)


  with t.no_grad():
    v_T = a2c.critic(obs).squeeze(-1)          # drop the * masks[-1]

  # obs contains s_{t+k}
  discounted_rewards = t.zeros(n_rollout, n_envs, device=device)
  for i in reversed(range(n_rollout)):
    discounted_rewards[i] = ep_rewards[i]

    if i + 1 < n_rollout:
      discounted_rewards[i] += df*discounted_rewards[i+1]*masks[i]
    else:
      discounted_rewards[i] += df * v_T * masks[i]

  w_loss = t.zeros((), device=device)
  p_loss = t.zeros((), device=device)

  for i in range(n_rollout):
    with t.no_grad():
      delta = discounted_rewards[i] - ep_state_values[i]

    w_loss = w_loss - (delta*ep_state_values[i]).mean()
    p_loss = p_loss - (delta * ep_action_log_probs[i]).mean() - 1e-5 * ep_entropy[i].mean()

  a2c.update_params(p_loss / n_rollout, w_loss / n_rollout)
  for s in scheds: s.step()

  if update % 10 == 0:
    pbar.set_postfix(
      actor=f"{p_loss.item():.3f}",
      critic=f"{w_loss.item():.3f}",
      ret=f"{np.mean(recent_returns):.1f}" if recent_returns else "—",
      refresh=False,
    )
    

In [ ]:
env = gym.make("LunarLander-v3", continuous=False, render_mode="human", max_episode_steps=1000)

with t.no_grad():
  obs, info = env.reset()
  obs = t.tensor(obs, device=device).unsqueeze(0)

  done = False
  total_rewards = 0
  while not done:
    # this is where you would insert your policy
    action, _, _, _ = a2c.select_action(obs)

    next_obs, reward, terminated, truncated, info = env.step(action.item())
    obs = t.tensor(next_obs, device=device).unsqueeze(0)
    done = terminated or truncated

    total_rewards += float(reward)

    if done:
      break

env.close()

total_rewards

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

returns = np.array(envs_wrapper.return_queue).flatten()
lengths = np.array(envs_wrapper.length_queue).flatten()

def smooth(x, w=50):
    return np.convolve(x, np.ones(w) / w, mode="valid") if len(x) >= w else x

fig, ax = plt.subplots(1, 3, figsize=(16, 4))

ax[0].plot(returns, alpha=0.25, lw=0.5, color="tab:blue")
ax[0].plot(np.arange(len(smooth(returns))) + 25, smooth(returns), color="tab:blue")
ax[0].axhline(0, color="k", lw=0.5)
ax[0].set_title("episode return (raw + smoothed)")
ax[0].set_xlabel("episode")

ax[1].plot(smooth(lengths), color="tab:orange")
ax[1].set_title("episode length")
ax[1].set_xlabel("episode")

w = 200
stds = [returns[i:i+w].std() for i in range(0, len(returns) - w, w // 4)]
ax[2].plot(np.arange(len(stds)) * (w // 4), stds, color="tab:green")
ax[2].set_title(f"return std (window={w})")
ax[2].set_xlabel("episode")

plt.tight_layout()
plt.show()

print(f"episodes: {len(returns)}")
print(f"peak smoothed: {smooth(returns).max():.1f}")
print(f"last-100 mean: {returns[-100:].mean():.1f}  std: {returns[-100:].std():.1f}")